# 00 · 시작하기

이 노트 묶음의 목표는 **MPC와 MPPI를 처음부터 배워서, 연구 프로젝트 "유효질량 기반 사람 회피 MPC"의 제어·비용을 직접 설계할 수 있게 되는 것**이다. 노트만 순서대로 따라가면 되도록 만들었고, 다른 자료는 필요 없다.

### 연구 프로젝트

협동로봇이 **움직이는 사람을 실시간으로 회피**하는 MPC(MPPI) 연구다. 사람의 체형·자세·충돌 방향에 따라 달라지는 **유효질량**을 MPC 비용에 넣는다. 예상 충돌력이 큰 부위·방향·자세일수록 로봇이 스스로 더 멀리 돌아가는 경로를 고르게 하는 것이 목표다. Isaac Sim에서 설계한 뒤 실제 Franka로 옮긴다(sim2real).

| 확정된 설계 | 내용 |
|---|---|
| 회피 방식 | 속도를 줄이는 것이 아니라 **경로를 바꿔 이격을 키우는 우회** |
| 질량 | 사람·로봇 유효질량의 환산질량 $\mu = m_h m_r / (m_h + m_r)$ |
| 충돌 방향 u | 로봇 점에서 가장 가까운 사람 분절 무게중심으로 향하는 단위벡터. 사람 유효질량은 u, 로봇은 −u 방향 |
| 비용 설계 순서 | 조합 1(방향별 환산질량 → 활성 거리) → 조합 2(ROM 반영 + 필요 이격 거리) |
| 제어 | cuRobo MPPI (후보 400, horizon 30, 반복 5) |

연구 기록: [노션 · 유효질량 기반 사람 회피 MPC](https://app.notion.com/p/419fcf2876fa806593b9d49d7d07a0ce) · 선행 프로젝트: [franka-depth-mppi](https://github.com/ukhyeon/franka-depth-mppi) ([노션](https://app.notion.com/p/3fcfcf2876fa80fa87d6efadd04559dc))

### 공부하는 것과 하지 않는 것

- **공부 대상:** MPC·MPPI 제어 설계와 그 이론적 이해. 특히 **"비용을 어떻게 설계하면 MPPI가 어떻게 행동하는가."**
- **다루지 않는 것:** 유효질량과 관성 행렬의 유도(관성 행렬은 주어진 값으로만 쓴다), 로봇 동역학, Pinocchio·cuRobo 구현 방법, MPC 안정성 이론, MPPI 경로 적분 유도.

**필요한 배경:** 위치·속도·가속도, 파이썬과 numpy 기초. 미분은 "기울기"라는 뜻만 알면 된다.

## 1. 지도

```text
A. 기초 ─ 1D 카트 하나로 시작
  01 로봇을 움직인다는 것        상태·입력·모델·rollout
  02 좋은 움직임을 숫자로        비용 함수
  03 가장 좋은 행동열 찾기       최적화·경사하강·국소 최솟값
B. MPC
  04 MPC: 계획하고 한 발만       receding horizon·warm start
C. MPPI
  05 샘플링으로 최적화하기 ★     MPPI의 핵심 (softmax 가중치·온도 β)
  06 MPPI로 장애물 피하기        2D·장애물 비용·할인·BEST·soft cost
  07 실전 MPPI의 디테일          매끄러운 노이즈·Halton·특수 후보·분산 갱신
D. 로봇팔
  08 점에서 로봇팔로             관절 공간·순기구학·충돌 구·η 비용
  09 cuRobo로 Franka 계획하기    실제 라이브러리의 rollout과 비용
E. 선행 프로젝트 franka-depth-mppi ─ 정적 손, 가정 질량 → 활성 거리
  10 로봇은 세상을 어떻게 보나    깊이 영상·지도·거리장·미관측
  11 손 전용 충돌 비용과 유효질량  비용 항 추가와 질량 → η 정책
  12 실험 결과 읽기              R / LOW / HIGH 비교
  13 Isaac Sim 실습              물리 시뮬레이터에서 직접 돌리기
F. 새 연구의 비용·제어 설계 ─ 평면 사람 장면, 요인 하나씩
  14 움직이는 사람               rollout 30칸에 사람을 어디에 두나
  15 η와 w                      질량을 범위로 넣을까, 세기로 넣을까
  16 방향별 질량                 로봇은 어느 쪽으로 비켜 가나
  17 ESS                        MPPI는 비용 차이를 얼마나 세밀하게 옮기나
  18 형상 비용 합치기            min·sum·max와 몸통 그림자
  19 필요 이격 거리              질량 → 활성 거리 대신 물리 기반 거리
  20 안전 필터                   소프트 비용이 못 지키는 이격 지키기
  21 지연과 보간                 60 Hz 계획 → 1 kHz 제어, 인지 지연
  22 cuRobo·Franka 확인          평면 장면의 현상이 실제로도 나오나
```

**A~E (01~13): 기초와 선행 프로젝트.** MPC와 MPPI를 처음부터 익히고, 그 위에 만든 선행 프로젝트 franka-depth-mppi를 읽는다. 선행 프로젝트는 정적인 손 하나에 **가정한 질량을 활성 거리 η로 바꾸는 휴리스틱**을 썼다. 새 연구는 이것을 움직이는 전신, 계산한 방향별 질량, 물리 기반 거리로 넓힌다.

**F (14~22): 새 연구의 설계 실험.** 평면 위의 2D 점 로봇이 "평면 사람"(몸통 원 + 팔 캡슐)을 지나 목표로 가는 **같은 장면**을 끝까지 쓰고, 노트마다 **요인 하나만** 바꾼다. 그래서 결과 차이가 그 요인 때문이라고 읽을 수 있다. 각 노트 첫머리에 "이 노트의 질문", "최종 목표", "연구의 어느 결정과 연결되는가"가 있다.

### F. 노트별 질문과 연구 결정

| 노트 | 질문 | 연구에서 쓰이는 곳 | 상태 |
|---|---|---|---|
| [14](14_moving_obstacle.ipynb) | 장애물이 움직이면 미래 30칸 각각에서 장애물은 어디에 있는가 (정지 가정 vs 등속 예측, 예측이 틀릴 때) | 사람 예측 방식 결정 | 작성 완료 |
| [15](15_eta_vs_weight.ipynb) | 질량을 활성 거리 η(범위)로 넣는 것과 가중치 w(세기)로 넣는 것은 이격을 어떻게 다르게 바꾸는가 (균형 이격 $d_{eq} = \eta(1 - g/w)$) | 비용 설계 | 작성 완료 |
| [16](16_directional_mass.ipynb) | 방향마다 질량이 다르면 로봇은 어느 쪽으로 비켜 가는가 (방향별 η(u), 환산질량에 의한 대비 압축) | 조합 1 | 작성 완료 |
| [17](17_ess.ipynb) | MPPI는 비용 차이를 얼마나 세밀하게 행동으로 옮기는가 (σ·β·K와 유효 샘플 수 ESS, 잡음 편향, η 차이의 구별 해상도) | 결과 해석 | 작성 완료 |
| [18](18_shape_combine.ipynb) | 사람 몸의 여러 형상의 비용을 min·sum·max로 합치면 비용 지형과 MPPI 회피가 어떻게 달라지는가 (몸통 그림자, 연속성, 중복 계산, 프로젝트가 max를 쓰는 이유) | 비용 합치는 방식 | 작성 완료 |
| 19 | 질량을 필요 이격 거리 $d_{req}$로 바꾸면 무엇이 달라지는가. 접근 속도를 실제 값으로 쓰면 감속으로 비용을 줄이는 탈출구가 생기는가 | 조합 2, 접근 속도 고정 여부 | 작성 예정 |
| 20 | 소프트 비용이 못 지키는 이격을 CBF 안전 필터($h = d - d_{req} \ge 0$)로 어떻게 지키는가, 필터가 너무 강하면 어떻게 되는가 | 안전 구조 결정 | 작성 예정 |
| 21 | 60 Hz 계획 → 1 kHz 보간, 추종 지연, 인지 지연이 이격을 어떻게 줄이고, 상태 외삽으로 어떻게 보상하는가 | 실기 제어 구성 | 작성 예정 |
| 22 | 15~18의 현상이 cuRobo MPPI와 Franka에서도 나오는가 (Isaac Sim 없이, 움직이는 캡슐 하나) | 실제 구현 전 확인 | 작성 예정 |

### 시간이 부족할 때

| 목적 | 읽을 노트 |
|---|---|
| MPC만 | 01 → 02 → 03 → 04 |
| MPPI만 (MPC를 안다면) | 01 → 02 → 05 → 06 → 07 |
| 선행 프로젝트 구조만 | 05 → 08 → 10 → 11 → 12 |
| **새 연구 설계 (최소 준비)** | 05 → 06 → 07 → 08 → 11 → 14 → 15 → … → 22 |

새 연구 설계에 바로 들어가기 전에 꼭 필요한 것은 05(MPPI 갱신과 β), 06(η 장애물 비용, warm start, soft cost), 07(특수 후보, 분산 갱신), 08(η 비용식), 11(비용 항 하나 더하기, 질량 → η 정책)이다.

## 2. 이 노트를 공부하는 방법

노트는 학습 연구에서 효과가 확인된 방법을 따라 구성했다. 각 장치가 왜 있는지 알고 쓰면 효과가 더 좋다.

| 노트 안의 장치 | 어떻게 쓰나 | 왜 |
|---|---|---|
| **문제 → 아이디어 → 이름** 순서 | 새 용어가 나오기 전에 "무엇이 불편한지"를 먼저 겪는다 | 필요를 먼저 느끼면 해법이 더 잘 기억된다 |
| **예측해 보기** | 코드를 실행하기 **전에** 결과를 한 줄 적어 본다. 틀려도 된다 | 예측과 결과의 차이가 가장 많이 배우는 순간이다 |
| **직접 해보기** | 값을 하나 바꿔 다시 실행한다. 예상 결과는 접어 두었다 | 스스로 조작해 봐야 이해가 손에 붙는다 |
| **한 문장으로** | 방금 배운 것을 자기 말로 한 문장 설명한다 | 설명하다 막히는 곳이 아직 모르는 곳이다 |
| **복습 퀴즈 / 확인 문제** | 답(접힌 부분)을 열기 전에 먼저 떠올린다. 14~22는 계산·보기 고르기처럼 답이 정해진 문제다 | 다시 읽기보다 떠올리기가 기억을 훨씬 오래 남긴다 |
| **같은 예제의 반복** | 1D 카트 → 2D 점 → 2관절 팔 → Franka, 14~22는 같은 평면 사람 장면 | 익숙한 예제 위에 새 개념 하나씩만 얹는다 |
| **요인 하나씩** (14~22) | 노트마다 바꾼 요인이 무엇인지 첫머리에서 확인한다 | 결과 차이를 그 요인 탓으로 읽을 수 있어야 연구 결정에 쓸 수 있다 |

**권장:** 하루에 노트 2~3개. 다음 날 첫 노트의 복습 퀴즈부터 시작한다(간격을 두고 떠올리기).

14~22의 "핵심 장면"은 미리 세운 가설이다. 현상이 나오지 않으면 이야기를 맞추지 않고 그대로 적는다(예: 14의 정면 접근은 정지 가정과 등속 예측의 차이가 작았다). 설계 근거와 참고 자료는 [`_design/curriculum.md`](_design/curriculum.md)에 정리했다.

## 3. 용어집

노트를 읽다가 용어가 헷갈리면 여기로 돌아온다. "처음"은 그 용어를 문제부터 설명하는 노트다. 15~22의 용어는 설계 기준이며, 노트를 쓰면서 표현이 바뀔 수 있다.

| 용어 | 영어 | 쉬운 뜻 | 처음 |
|---|---|---|---|
| 상태 | state | 미래를 계산하는 데 필요한 지금 정보. 카트는 (위치, 속도) | 01 |
| 입력 | control input | 우리가 매 순간 정하는 값. 이 노트에서는 가속도 | 01 |
| 모델 | model | 상태와 입력으로 다음 상태를 계산하는 규칙 | 01 |
| rollout | rollout | 입력열을 모델에 차례로 넣어 미래 상태열을 계산하는 일 | 01 |
| horizon | horizon | 몇 스텝 앞까지 내다볼지. T스텝 = T·Δt초 | 01 |
| 비용 | cost | 움직임이 얼마나 나쁜지를 나타내는 숫자. 작을수록 좋다 | 02 |
| 최종 비용 | terminal cost | horizon 마지막 상태에만 주는 비용 (끝에 목표에 있나, 멈췄나) | 02 |
| 최적화 | optimization | 비용이 가장 작은 입력열을 찾는 일 | 03 |
| 경사하강 | gradient descent | 비용의 기울기 반대 방향으로 조금씩 이동하는 최적화 | 03 |
| 국소 최솟값 | local minimum | 주변보다는 낮지만 가장 낮지는 않은 골짜기 | 03 |
| MPC | Model Predictive Control | 매 스텝 계획을 세우고 첫 입력만 실행한 뒤 다시 계획하는 제어 | 04 |
| receding horizon | receding horizon | 시간이 지나면 예측 구간도 함께 앞으로 밀려가는 방식 | 04 |
| warm start | warm start | 직전 계획을 한 칸 당겨 다음 최적화의 출발점으로 쓰는 것 | 04 |
| 샘플 / 후보 | sample / particle | 무작위로 만든 입력열 하나. cuRobo는 particle이라 부른다 | 05 |
| softmax 가중치 | softmax weight | 비용이 낮을수록 크게, 합이 1이 되게 만든 가중치 | 05 |
| 온도 β | temperature | 비용 차이를 얼마나 엄격하게 볼지. 작으면 최선 하나에 집중 | 05 |
| MPPI | Model Predictive Path Integral | 평균 주변 샘플 → 비용 → softmax 가중평균을 반복하는 최적화 | 05 |
| 탐색 폭 σ | standard deviation | 평균 주변에 후보를 얼마나 넓게 뿌릴지 | 05 |
| activation distance η | activation distance | 장애물까지 이 거리 안으로 들어오면 비용이 켜진다 | 06, 08 |
| 할인 γ | discount | 먼 미래의 비용을 덜 세기 위해 곱하는 값 (0.98^t) | 06 |
| BEST | best sample | 가중평균 대신 가중치가 가장 큰 후보 하나를 명령으로 쓰는 방식 | 06 |
| soft cost | soft constraint | 금지가 아니라 비싸게 만드는 방식. 위반 후보도 선택될 수 있다 | 06 |
| 매끄러운 노이즈 | knot noise | 몇 개 점만 뽑고 곡선으로 이어 만든 부드러운 무작위 입력 | 07 |
| Halton 수열 | Halton sequence | 무작위처럼 보이지만 공간을 더 고르게 채우는 수열 | 07 |
| 관절 공간 / 작업 공간 | joint / task space | 관절 각도로 본 로봇 / 손끝 위치로 본 로봇 | 08 |
| 순기구학 | forward kinematics (FK) | 관절 각도 → 손끝·링크 위치 계산 | 08 |
| 충돌 구 | collision sphere | 로봇 모양을 구 여러 개로 근사한 것. 거리 계산이 쉽다 | 08 |
| buffer | sphere buffer | 충돌 구 반지름에 더하는 여유 (선행 프로젝트 20 mm) | 08, 09 |
| 깊이 영상 | depth image | 픽셀마다 카메라까지의 거리가 담긴 영상 | 10 |
| voxel | voxel | 3D 공간을 나눈 작은 정육면체 칸 (선행 프로젝트 3 cm) | 10 |
| 거리장 (ESDF) | Euclidean signed distance field | 공간의 각 점에서 가장 가까운 장애물까지 거리를 저장한 지도 | 10 |
| 미관측 | unknown | 카메라가 보지 못한 곳. 비어 있다는 뜻이 아니다 | 10 |
| ROI | region of interest | 깊이 영상에서 손을 찾을 관심 영역 | 11 |
| 환산 질량 μ | reduced mass | 두 물체가 부딪힐 때 실제로 작용하는 유효 질량 $m_h m_r/(m_h+m_r)$ | 11 |
| 예측 | prediction | 지금 관측으로 rollout의 미래 각 칸에서 사람이 어디 있을지 정하는 규칙 | 14 |
| 정지 가정 | static assumption | 모든 미래 칸에서 사람이 지금 자리에 있다고 보는 예측 | 14 |
| 등속 예측 | constant velocity | 사람이 지금 속도 그대로 계속 움직인다고 보는 예측 | 14 |
| 오라클 | oracle | 참 미래를 아는 비교 기준. 실제로는 쓸 수 없다 | 14 |
| 방향 유효질량 | directional effective mass | 방향 u로 밀 때의 질량 $m_h(\mathbf{u}) = 1/(\mathbf{u}^\top M^{-1}\mathbf{u})$ | 15, 16 |
| 충돌 방향 u | collision direction | 로봇 점 → 가장 가까운 사람 분절 무게중심 단위벡터 | 16 |
| ROM 잠금 | range-of-motion lock | 관절이 가동 범위 끝에 걸려 한쪽으로 밀 때 더 무거워지는 것 | 16 |
| 대비 압축 | contrast compression | 사람 질량 차이가 환산질량에서는 작아지는 현상 | 16 |
| 균형 이격 | equilibrium clearance | 목표가 끄는 힘과 사람 비용이 미는 힘이 같아지는 거리 $d_{eq}$ | 15 |
| 유효 샘플 수 ESS | effective sample size | $(\sum w)^2/\sum w^2$. 가중치가 하나에 몰리면 1, 고르게 퍼지면 K | 17 |
| 떨림 | chattering | 명령이 스텝마다 크게 바뀌는 것. 명령 변화량으로 잰다 | 17, 18 |
| 몸통 그림자 | torso shadow | 가벼운 형상(손)이 가장 가까워, 바로 뒤 무거운 형상(몸통)의 더 큰 비용이 min 합치기에서 빠지는 것 | 18 |
| 중복 계산 | double counting | 같은 몸이 형상 여러 개에 들어가, sum 합치기에서 비용이 여러 번 더해지는 것 | 18 |
| 필요 이격 거리 | required separation distance | 반응 거리 + 제동 거리 + 여유로 정한, 사람과 떨어져야 할 거리 $d_{req}$ | 19 |
| 허용 속도 | allowable speed | 충돌해도 허용 힘을 넘지 않는 접근 속도 $v_{allow}$ | 19 |
| 안전 필터 (CBF) | control barrier function filter | MPC 명령을 $h \ge 0$이 지켜지도록 최소한으로 고치는 장치 | 20 |
| 보간 | interpolation | 60 Hz 계획 사이를 1 kHz 명령으로 채우는 일 | 21 |
| 상태 외삽 | state extrapolation | 지연된 측정으로부터 지금 상태를 앞당겨 추정하는 일 | 21 |

## 4. 기호표

| 기호 | 뜻 | 단위 / 예 |
|---|---|---|
| $p$, $v$ | 위치, 속도 | m, m/s |
| $x = (p, v)$ | 상태 | |
| $u$ | 입력(가속도) | m/s² (로봇팔은 rad/s²) |
| $\Delta t$ | 한 스텝의 시간 | 0.1 s (01~05), 0.05 s (06~07, 14~) |
| $T$ | horizon 길이(스텝 수) | 20, 30 |
| $t$, $\tau_t$ | 시점 번호, 지금부터 칸 t까지의 시간 $(t+1)\Delta t$ | |
| $U = (u_0, \dots, u_{T-1})$ | 입력열 | 배열 모양 `[T]`, 후보 K개면 `[K, T]` |
| $J(U)$ | 입력열 $U$의 총비용 | |
| $c_t$ | 시점 $t$의 비용 | |
| $w_{\text{goal}}, w_u, w$ | 비용 항의 가중치 | (샘플 가중치 ω와 다르다) |
| $K$, $k$ | 후보 수, 후보 번호 | 400 |
| $\omega_k$ | 후보 $k$의 softmax 가중치 | 합이 1 |
| $\beta$ | 온도 | 1.0 (05~07, 14~), 0.1 (cuRobo) |
| $\gamma$ | 할인 | 0.98 (cuRobo) |
| $\mu$, $\sigma$ | 평균 입력열, 탐색 폭(표준편차) | (환산질량 μ와 문맥으로 구분) |
| $\eta$ | activation distance | 0.10 m (14), 25 / 73.52 mm (선행 프로젝트) |
| $d$ | 로봇 원 표면과 사람 분절 표면 사이 이격 | m |
| $\mathbf{u}$ (굵게) | 충돌 방향 단위벡터 | |
| $M$ | 분절의 2×2 관성 행렬 (주어진 값) | kg |
| $m_h(\mathbf{u})$, $m_r$ | 사람 방향 유효질량, 로봇 유효질량 | kg, $m_r$ = 5 kg |
| $\mu = m_h m_r/(m_h+m_r)$ | 환산질량 | kg |
| $d_{eq}$, $g$ | 균형 이격, 목표가 끄는 기울기 | m, (15) |
| $d_{req}$, $v_n$, $T_r$, $a_b$, $d_0$ | 필요 이격 거리, 접근 속도, 반응 시간, 감속도, 여유 | (19) |
| $v_{allow}$, $F_{max}$, $k$ | 허용 속도, 허용 힘, 부위 강성 | (19) |
| $h = d - d_{req}$ | 안전 필터가 지키는 값 ($h \ge 0$) | (20) |

## 5. 실행 환경

모든 노트에는 실행 결과와 그림이 이미 저장되어 있어서 **실행하지 않고 읽기만 해도 된다.** 하지만 "예측해 보기"와 "직접 해보기"는 직접 실행할 때 효과가 있다.

- **커널:** VSCode에서 노트를 열면 **"Isaac Sim 4.5 + cuRobo"** 커널이 자동으로 선택된다. 목록에 없으면 커널 선택 → "Jupyter Kernel..."에서 고른다.
  - 이 커널은 Isaac Sim 내장 파이썬(`~/isaacsim/python.sh`, Python 3.10)으로 실행된다. conda 가상환경 `isaacsim`과는 다르다.
  - 커널 설정은 [`kernel_launcher.py`](kernel_launcher.py)와 `~/.local/share/jupyter/kernels/isaacsim-curobo/kernel.json`에 있다. ipykernel은 `.kernel_pkgs/`에 따로 설치해서 Isaac Sim 자체는 바뀌지 않았다.
- **GPU가 필요한 노트:** 09(cuRobo), 13의 실행 셀(Isaac Sim), 22(cuRobo, 작성 예정). 나머지는 numpy·scipy·matplotlib만 쓰고 CPU로 돈다. 14~21은 CPU로만 실행한다.
- **공용 파일**

| 파일 | 쓰는 노트 | 내용 |
|---|---|---|
| `common.py` | 01~13 | 경로·그림 설정·cuRobo 식 일부 |
| `course.py` | 01~13 | 01~07에서 직접 만든 코드의 정리본 |
| `common14/` | 14~22 | 평면 사람 장면(수치 고정), 2D 점 MPPI, 분절 질량, 지표. 수치와 이유는 [`common14/README.md`](common14/README.md) |

- 이전 버전 노트(복습용, 노션 기록 대조 포함)는 `archive_v1/`에 있다.

아래 셀로 환경을 확인할 수 있다.

In [1]:
from common import check_paths
check_paths()
import importlib
for m in ['numpy', 'scipy', 'matplotlib', 'yaml', 'common14', 'torch', 'curobo']:
    try:
        mod = importlib.import_module(m)
        print(f'{m:11s} OK  {getattr(mod, "__version__", "")}')
    except ImportError:
        print(f'{m:11s} 없음 (09, 13, 22에서만 필요)' if m in ('torch', 'curobo') else f'{m:11s} 없음')

REPO      OK  ~/franka-depth-mppi
CUROBO    OK  ~/curobo
TAPE      OK  ~/franka-depth-mppi/data/hand-tape-20261012-000004-472steps-registered
EVIDENCE  OK  ~/franka-depth-mppi/evidence
numpy       OK  1.26.0
scipy       OK  1.15.3
matplotlib  OK  3.8.4
yaml        OK  6.0.1
common14    OK  
torch       OK  2.5.1+cu118
curobo      OK  0.7.7.post1.dev0+dirty


**시작:** [01 · 로봇을 움직인다는 것](01_state_input_model.ipynb) · 새 연구 설계부터 보려면 [14 · 움직이는 사람](14_moving_obstacle.ipynb)